# single structure

In [ ]:
from GNBO.crystal.Find_Average_Structure import find_average_structure
from pymatgen.io.cif import CifWriter
from pymatgen.io.cif import CifParser
import warnings
warnings.simplefilter("ignore")
# input structure(unrelaxed structure + relaxed supercell)
parser = CifParser("Your//cif//of//unrelaxed//structure//.cif")
structure_unrelax = parser.parse_structures(primitive = False, check_occu = False)[0]
parser = CifParser("Your//cif//of//relaxed//structure//final_structure.cif")
structure_relax = parser.parse_structures(primitive = False, check_occu = False)[0]
target_comp = [] #dict of composition, for example: (Ca0.66La0.34)(Ti0.66Al0.34)O3 → [{"Ca":0.66, "La":0.34},{"Ti":0.66, "Al":0.34}]
spn = int(62)  # space group number
output_path = "Your//output//path//"
sm = (8, 8, 8)  # n × n × n of supercell

In [ ]:
sqs_finder = find_average_structure(
    target=target_comp, superlattice_relax=structure_relax, target_sp=spn, output=output_path
)
average_structure = sqs_finder.get_average_structure(structure_unrelax, sm, True)
refined_structure = sqs_finder.get_refined_structure(structure_unrelax, sm, True)

# Batch analysis

In [ ]:
from glob import glob
import os
import pandas as pd
import re
# from pymatgen.core import Structure
from pymatgen.io.cif import CifParser
from GNBO.crystal.Find_Average_Structure import find_average_structure
import warnings
warnings.simplefilter("ignore")

spn_data = pd.read_csv("path//of//spn//spn.csv")
datas = glob("glob//path//of//relaxed//structure//*")
dire_unrelax = "path//of//unrelaxed//structure//"
outputpath = "output//path//of//result//structure/"

In [ ]:
def split_chemical_formula(formula):
    bracket_contents = re.findall(r'\((.*?)\)', formula)
    non_bracket_str = re.sub(r'\([^)]*\)', '', formula)
    only_letters = re.sub(r'[0-9.]', '', non_bracket_str)
    elements = re.findall(r'[A-Z][a-z]*', only_letters)
    final_result = bracket_contents + elements
    return final_result
def str_to_chem_dict(chem_str):
    pattern = r'([A-Z][a-z]*)(\d+(\.\d+)?)?'

    matches = re.findall(pattern, chem_str)

    chem_dict = {}
    for elem, value, _ in matches:
        if elem:
            chem_dict[elem] = float(value) if value else 1.0
    
    return chem_dict

def split_repeated_num_str(num_str):
    try:
        if not num_str.isdigit():
            raise ValueError(f"Input'{num_str}'is not a pure numeric string")

        str_len = len(num_str)
        if str_len % 3 != 0:
            raise ValueError(
                f"Input'{num_str}'Length is{str_len},is not a multiple of 3, does not meet 'repeat three times' rule"
            )
        
        unit_len = str_len // 3
        
        unit1 = num_str[0:unit_len]
        unit2 = num_str[unit_len:2*unit_len]
        unit3 = num_str[2*unit_len:3*unit_len]
        
        if not (unit1 == unit2 == unit3):
            raise ValueError(
                f"Input'{num_str}'Split units are inconsistent: {unit1}, {unit2}, {unit3}"
            )
        
        num_unit = int(unit1)
        result_tuple = (num_unit, num_unit, num_unit)
        
        return result_tuple
    
    except ValueError as e:
        print(f"Error: {e},Cannot convert to target tuple")
        return ()
    except Exception as e:
        print(f"Unknown error: {str(e)}")
        return ()

In [ ]:
fail_data = pd.DataFrame()
for data in datas:
    # dopant dict
    name = os.path.basename(data)
    names = name.split("_")
    full_path = os.path.join(data, "average_structure.cif")
    if os.path.isfile(full_path):
        print(f"{names[0]}fetched")
    else:
        print(f"Fetching{names[0]}")
        formula_ratio = split_chemical_formula(names[0])
        target_comp = []
        for fr in formula_ratio:
            formula = str_to_chem_dict(fr)
            if len(formula) > 1:
                target_comp.append(formula)
        # space group number
        sp_symbol = names[1]
        for row in spn_data.values.tolist():
            space_group, space_group_number = row
            if int(space_group_number) == int(sp_symbol):
                spn = space_group_number

        # output direction
        output_path = os.path.join(outputpath, name)
        # supercell
        for file_name in os.listdir(data):
            if file_name.endswith('.cif'):
                if len(file_name) > 0 and file_name[0].isdigit():
                    super_disorder_unrelax_structure = file_name[:-4]
        super_disorder_matrix = super_disorder_unrelax_structure.split("_")[0]
        sm = split_repeated_num_str(super_disorder_matrix)
        # relaxed structure
        structure_relax_dire = os.path.join(data, "disorder", "final_structure.cif")
        parser = CifParser(structure_relax_dire)
        structure_relax = parser.parse_structures(primitive = False, check_occu = False)[0]
        # unrelaxed structure
        structure_unrelax_name = f"{names[0]}_{names[1]}.cif"
        parser = CifParser(os.path.join(dire_unrelax, structure_unrelax_name))
        structure_unrelax = parser.parse_structures(primitive = False, check_occu = False)[0]
        # symmetry analysis
        sqs_finder = find_average_structure(target=target_comp, superlattice_relax=structure_relax, target_sp=spn, output=output_path)
        """
        try:
            average_structure = sqs_finder.get_average_structure(structure_unrelax, sm, True)
            print("Average structure fetched successfully")
            refined_structure = sqs_finder.get_refined_structure(structure_unrelax, sm, True)
            print("Symmetry structure fetched successfully")
        except:
            print(f"{names[0]}Fetch failed")
            fail_data = fail_data._append({"fromula":names[0]}, ignore_index = True)
        """
        average_structure = sqs_finder.get_average_structure(structure_unrelax, sm, True)
        refined_structure = sqs_finder.get_refined_structure(structure_unrelax, sm, True)

FetchingBaTiO3


ValueError: invalid literal for int() with base 10: 'P4mm'